In [1]:
from google.colab import files
uploaded = files.upload()

Saving spotify_history.csv to spotify_history.csv
Saving spotify_recommendations.csv to spotify_recommendations.csv


In [2]:
"""
Cosine Similarity
Dataset: spotify_history.csv
Predicts: Songs similar users liked that the target user hasn't heard
Features: Per-user listening vectors (skip-adjusted play weights per track)

"""

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# 1. Load Data
history = pd.read_csv("spotify_history.csv", parse_dates=["ts"])
print(f"Raw history shape: {history.shape}")

# 2. Simulate multi-user dataset
# Each participant's file gets a unique user_id column.
# Partition the single-user history chronologically into 25 synthetic users to
# demonstrate the full collaborative filtering pipeline.

np.random.seed(42)
history = history.sort_values("ts").reset_index(drop=True)
n_users = 25

# Split rows into 25 roughly equal partitions (simulates 25 participant files)
history["user_id"] = pd.cut(history.index, bins=n_users,
                             labels=[f"user_{i:02d}" for i in range(n_users)])
history["user_id"] = history["user_id"].astype(str)
print(f"Simulated {n_users} users | Users: {history['user_id'].nunique()}")
print(f"Sample:\n{history[['user_id','track_name','artist_name','ms_played','skipped']].head(3).to_string()}\n")

# 3. Compute skip-adjusted play rates
# Raw play count is noisy — background listening inflates it.
# We build a weighted score per (user, track) that rewards completed plays and replay and penalizes skips (the strongest negative signal)
# Weight formula:
# weight = (complete_plays * 2 + non_skip_plays * 1 - skip_plays * 1.5) / total_plays
# Set to [0, 1] and set to 0 if only 1 play occurred (not enough signal)

SKIP_REASONS = {"fwdbtn", "backbtn", "nextbtn"}
h = history.copy()
h["is_skip"]     = h["skipped"] | h["reason_end"].isin(SKIP_REASONS)
h["is_complete"]  = h["reason_end"] == "trackdone"

interaction = h.groupby(["user_id", "spotify_track_uri"]).agg(
    play_count     = ("ms_played",    "count"),
    skip_count     = ("is_skip",      "sum"),
    complete_count = ("is_complete",  "sum"),
).reset_index()

interaction["non_skip_count"] = interaction["play_count"] - interaction["skip_count"]

interaction["weight"] = (
    (interaction["complete_count"] * 2 +
     interaction["non_skip_count"] * 1 -
     interaction["skip_count"]     * 1.5)
    / interaction["play_count"]
).clip(lower=0)

# Zero-out single-play tracks: insufficient evidence of preference
interaction.loc[interaction["play_count"] == 1, "weight"] = 0

# Binary "liked" label: weight > 0.5 threshold
interaction["liked"] = (interaction["weight"] > 0.5).astype(int)

print(f"Interaction table shape: {interaction.shape}")
print(f"Like rate: {interaction['liked'].mean():.2%}\n")

# 4. Build user-item matrix
# Rows = users, Columns = track_uris, Values = skip-adjusted weights
user_item_matrix = interaction.pivot_table(
    index="user_id",
    columns="spotify_track_uri",
    values="weight",
    fill_value=0
)
print(f"User-item matrix: {user_item_matrix.shape[0]} users × {user_item_matrix.shape[1]} tracks")
print(f"Sparsity: {(user_item_matrix == 0).values.mean():.1%}\n")

# 5. Computer user-user cosine similarity
# Cosine similarity is scale-invariant: a user with 50 tracks and one with
# 500 tracks can still have a meaningful similarity score.
similarity_matrix = cosine_similarity(user_item_matrix)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

# Plot similarity heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(similarity_df, cmap="Greens", xticklabels=False, yticklabels=False)
plt.title("User-User Cosine Similarity Matrix")
plt.tight_layout()
plt.savefig("model3_similarity_heatmap.png", dpi=150)
plt.close()
print("Saved: model3_similarity_heatmap.png")

# 6. Recommendation function
def recommend_songs_collab(user_id: str, user_item_matrix: pd.DataFrame,
                            similarity_df: pd.DataFrame,
                            top_k_users: int = 10,
                            top_n_songs: int = 10) -> pd.DataFrame:
    """
    Recommend songs to a target user based on what similar users enjoyed.

    Args:
        user_id:           Target user to generate recommendations for
        user_item_matrix:  User × track weight matrix
        similarity_df:     Precomputed user-user cosine similarity DataFrame
        top_k_users:       Number of most similar users to consider
        top_n_songs:       Number of songs to recommend

    Returns:
        DataFrame of recommended track URIs with scores.
    """
    if user_id not in similarity_df.index:
        raise ValueError(f"User '{user_id}' not found in similarity matrix.")

    # Get top-K most similar users (excluding the current user)
    sim_scores = similarity_df[user_id].drop(index=user_id).sort_values(ascending=False)
    top_k = sim_scores.head(top_k_users)

    # Tracks the target user has already heard (weight > 0)
    target_tracks = set(user_item_matrix.columns[user_item_matrix.loc[user_id] > 0])

    # Aggregate weighted scores across top-K users
    # Songs heard by more similar users get a higher score
    candidate_scores = {}
    for similar_user, sim_score in top_k.items():
        user_tracks = user_item_matrix.loc[similar_user]
        liked_tracks = user_tracks[user_tracks > 0.5].index  # similar user's liked songs

        for track in liked_tracks:
            if track not in target_tracks:  # only recommend unseen songs
                candidate_scores[track] = candidate_scores.get(track, 0) + sim_score

    if not candidate_scores:
        return pd.DataFrame(columns=["spotify_track_uri", "score", "n_similar_users"])

    recs = pd.DataFrame([
        {"spotify_track_uri": t, "score": s}
        for t, s in candidate_scores.items()
    ]).sort_values("score", ascending=False).head(top_n_songs)

    # Count how many of the top-K users liked each recommended song
    recs["n_similar_users"] = recs["spotify_track_uri"].apply(
        lambda t: sum(1 for u in top_k.index if user_item_matrix.loc[u, t] > 0.5
                      if t in user_item_matrix.columns)
    )
    return recs.reset_index(drop=True)

# 7. Leave-one-out evaluation
# Hide the last 20% of their liked tracks, generate recommendations, check
# how many hidden tracks appear in the top-N recommendations (for each user).

def evaluate_collaborative(user_item_matrix, similarity_df, top_k=10, top_n=10):
    precisions, recalls, f1s = [], [], []

    for user_id in user_item_matrix.index:
        liked = user_item_matrix.columns[user_item_matrix.loc[user_id] > 0.5].tolist()
        if len(liked) < 5:
            continue  # skip users with too few liked songs for meaningful eval

        # Hold out 20% of liked tracks as ground truth
        n_holdout = max(1, int(len(liked) * 0.2))
        np.random.shuffle(liked)
        holdout = set(liked[:n_holdout])
        train_tracks = liked[n_holdout:]

        # Temporarily zero out holdout tracks for this user
        eval_matrix = user_item_matrix.copy()
        eval_matrix.loc[user_id, list(holdout)] = 0

        # Recompute similarity with modified matrix
        sim = cosine_similarity(eval_matrix)
        sim_df = pd.DataFrame(sim, index=eval_matrix.index, columns=eval_matrix.index)

        recs = recommend_songs_collab(user_id, eval_matrix, sim_df, top_k, top_n)
        if recs.empty:
            continue

        recommended = set(recs["spotify_track_uri"])
        hits = recommended & holdout

        precision = len(hits) / len(recommended) if recommended else 0
        recall    = len(hits) / len(holdout) if holdout else 0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0)

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    return {
        f"Precision@{top_n}": np.mean(precisions),
        f"Recall@{top_n}":    np.mean(recalls),
        f"F1@{top_n}":        np.mean(f1s),
        "n_users_evaluated":  len(precisions),
    }

print("Running leave-one-out evaluation...")
eval_results = evaluate_collaborative(user_item_matrix, similarity_df, top_k=10, top_n=10)
print("\n── EVALUATION RESULTS ──")
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# 8. Sample recommendations for one user
target_user = user_item_matrix.index[0]
print(f"\n── SAMPLE RECOMMENDATIONS for {target_user} ──")
sample_recs = recommend_songs_collab(target_user, user_item_matrix, similarity_df)
print(sample_recs.to_string())

# 9. Top similar users visualization
sim_scores = similarity_df[target_user].drop(index=target_user).sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 4))
sim_scores.plot(kind="bar", color="#1DB954", edgecolor="white")
plt.xlabel("User ID")
plt.ylabel("Cosine Similarity")
plt.title(f"Top 10 Most Similar Users to {target_user}")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("model3_top_similar_users.png", dpi=150)
plt.close()
print("Saved: model3_top_similar_users.png")

# 10. Track coverage analysis
# How many unique tracks appear in at least 1 user's history?
coverage = (user_item_matrix > 0).sum(axis=0)
plt.figure(figsize=(8, 4))
plt.hist(coverage, bins=20, color="#1DB954", edgecolor="white")
plt.xlabel("Number of users who played this track")
plt.ylabel("Number of tracks")
plt.title("Track Coverage Across Users")
plt.tight_layout()
plt.savefig("model3_track_coverage.png", dpi=150)
plt.close()
print("Saved: model3_track_coverage.png")


Raw history shape: (145139, 20)
Simulated 25 users | Users: 25
Sample:
   user_id                                     track_name    artist_name  ms_played  skipped
0  user_00                            Say It, Just Say It   The Mowgli's       3185    False
1  user_00  Drinking from the Bottle (feat. Tinie Tempah)  Calvin Harris      61865    False
2  user_00                                    Born To Die   Lana Del Rey     285386    False

Interaction table shape: (61391, 8)
Like rate: 34.95%

User-item matrix: 25 users × 16342 tracks
Sparsity: 94.6%

Saved: model3_similarity_heatmap.png
Running leave-one-out evaluation...

── EVALUATION RESULTS ──
  Precision@10: 0.4680
  Recall@10: 0.0266
  F1@10: 0.0501
  n_users_evaluated: 25

── SAMPLE RECOMMENDATIONS for user_00 ──
        spotify_track_uri     score  n_similar_users
0  0JBvtprXP2Z0LP3jmzA7Xp  1.346101               10
1  7cX4PJz1old9fyFI8RlfgW  1.346101               10
2  7qbRP2jO0Sq3kTYdEW1v00  1.346101               10
3  3oJ